Qwen3 From Scratch (A Standalone Notebook)

In [ ]:
# 中文注释：检查运行本 notebook 所需的关键第三方库是否已安装，并打印其版本号，
# 便于排查因版本不一致导致的兼容性问题（例如 safetensors / tokenizers 的 API 变化）。
from importlib.metadata import version

# 中文注释：需要检查版本的依赖包列表
pkgs = [
    "huggingface_hub",  # to download pretrained weights
    "tokenizers",       # to implement the tokenizer
    "torch",            # to implement the model
]
# 中文注释：遍历并打印每个包当前安装的版本号
for p in pkgs:
    print(f"{p} version: {version(p)}")

In [ ]:
# 中文注释：通过下面三个布尔开关选择要加载的 Qwen3 模型变体，三者中必须且只能有一个为 True：
#   USE_BASE_MODEL      -> 预训练基座模型（无对话模板，未做指令/推理微调）
#   USE_REASONING_MODEL -> “思考”（reasoning/thinking）模型，生成时会输出 <think>...</think> 推理过程
#   USE_INSTRUCT_MODEL  -> 指令微调（chat/instruct）模型，直接输出对话回复，默认不含思考过程
# Select which model to use via the following flag; only one can be True

USE_BASE_MODEL = False
USE_REASONING_MODEL = True
USE_INSTRUCT_MODEL = False

# 中文注释：把三个布尔值当作 0/1 相加，校验有且仅有一个开关为 True，否则报错终止
if (USE_BASE_MODEL + USE_REASONING_MODEL
    + USE_INSTRUCT_MODEL) != 1:
    raise AttributeError("Only one of the options above can be True.")

1. Architecture code

In [ ]:
# 中文注释：本单元定义 Qwen3 的完整架构模块，从下到上依次是：
#   FeedForward（SwiGLU 前馈网络）、RMSNorm（均方根归一化）、
#   compute_rope_params / apply_rope（旋转位置编码 RoPE）、
#   GroupedQueryAttention（分组查询注意力 GQA，含可选的 QK-Norm）、
#   TransformerBlock（残差 + 前置归一化的 Transformer 层）、
#   Qwen3Model（词嵌入 + N 层 Transformer + 最终归一化 + 输出头）。
import torch
import torch.nn as nn


# 中文注释：SwiGLU 前馈网络（比标准 ReLU/GELU MLP 多一路“门控”分支）。
# fc1、fc2 都是 emb_dim -> hidden_dim 的线性层（分别对应 HF 权重里的 gate_proj、up_proj），
# fc3 是 hidden_dim -> emb_dim 的线性层（对应 down_proj），三者都不带偏置 bias。
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.fc1 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc2 = nn.Linear(cfg["emb_dim"], cfg["hidden_dim"], dtype=cfg["dtype"], bias=False)
        self.fc3 = nn.Linear(cfg["hidden_dim"], cfg["emb_dim"], dtype=cfg["dtype"], bias=False)

    # 中文注释：forward 输入 x 形状为 (batch, seq_len, emb_dim)。
    # SwiGLU 计算：down_proj( silu(gate_proj(x)) * up_proj(x) )，
    # 即用 silu(x_fc1) 作为“门控”逐元素乘以 x_fc2，再投影回 emb_dim 维度。
    def forward(self, x):
        x_fc1 = self.fc1(x)
        x_fc2 = self.fc2(x)
        x = nn.functional.silu(x_fc1) * x_fc2
        return self.fc3(x)
# 中文注释：RMSNorm（均方根归一化）。与 LayerNorm 的区别是不减均值、只按均方根缩放，
# 计算量更小且在大模型中效果与 LayerNorm 相当。qwen3_compatible=True 时会先升到 float32
# 计算方差再转回原 dtype，以匹配官方实现在低精度（如 bfloat16）下的数值稳定性。
class RMSNorm(nn.Module):
    def __init__(self, emb_dim, eps=1e-6, bias=False, qwen3_compatible=True):
        super().__init__()
        self.eps = eps
        self.qwen3_compatible = qwen3_compatible
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim)) if bias else None

    # 中文注释：x 形状可以是 (batch, seq_len, dim) 或按注意力头拆分后的 (batch, heads, seq_len, head_dim)，
    # 归一化统一在最后一维 dim=-1 上进行：
    #   variance = mean(x^2)；norm_x = x / sqrt(variance + eps) * scale (+ shift)
    def forward(self, x):
        input_dtype = x.dtype

        if self.qwen3_compatible:
            x = x.to(torch.float32)

        variance = x.pow(2).mean(dim=-1, keepdim=True)
        norm_x = x * torch.rsqrt(variance + self.eps)
        norm_x = norm_x * self.scale

        if self.shift is not None:
            norm_x = norm_x + self.shift

        return norm_x.to(input_dtype)
# 中文注释：预计算 RoPE（旋转位置编码）用到的 cos/sin 表。
# theta_base 越大，远距离位置的旋转角度差越小，有利于扩展到更长的上下文（Qwen3 用 1e6）。
# inv_freq 形状 (head_dim//2,)，为每一对维度分配一个不同的旋转频率。
def compute_rope_params(head_dim, theta_base=10_000, context_length=4096, dtype=torch.float32):
    assert head_dim % 2 == 0, "Embedding dimension must be even"

    # Compute the inverse frequencies
    inv_freq = 1.0 / (theta_base ** (torch.arange(0, head_dim, 2, dtype=dtype)[: (head_dim // 2)].float() / head_dim))

    # Generate position indices
    positions = torch.arange(context_length, dtype=dtype)

    # Compute the angles
    angles = positions.unsqueeze(1) * inv_freq.unsqueeze(0)  # Shape: (context_length, head_dim // 2)

    # Expand angles to match the head_dim
    angles = torch.cat([angles, angles], dim=1)  # Shape: (context_length, head_dim)

    # Precompute sine and cosine
    cos = torch.cos(angles)
    sin = torch.sin(angles)

    return cos, sin


# 中文注释：将 RoPE 旋转应用到 q/k 张量上。
# x 形状 (batch_size, num_heads, seq_len, head_dim)；
# 把最后一维切成前后两半 x1、x2，构造 rotated=(-x2, x1)，
# 再按 x*cos + rotated*sin 完成“旋转”，实现相对位置编码的效果。
def apply_rope(x, cos, sin):
    # x: (batch_size, num_heads, seq_len, head_dim)
    batch_size, num_heads, seq_len, head_dim = x.shape
    assert head_dim % 2 == 0, "Head dimension must be even"

    # Split x into first half and second half
    x1 = x[..., : head_dim // 2]  # First half
    x2 = x[..., head_dim // 2 :]  # Second half

    # Adjust sin and cos shapes
    cos = cos[:seq_len, :].unsqueeze(0).unsqueeze(0)  # Shape: (1, 1, seq_len, head_dim)
    sin = sin[:seq_len, :].unsqueeze(0).unsqueeze(0)

    # Apply the rotary transformation
    rotated = torch.cat((-x2, x1), dim=-1)
    x_rotated = (x * cos) + (rotated * sin)

    # It's ok to use lower-precision after applying cos and sin rotation
    return x_rotated.to(dtype=x.dtype)
# 中文注释：分组查询注意力（Grouped-Query Attention, GQA）。
# 与标准多头注意力不同：Query 仍有 num_heads 个头，但 Key/Value 只有更少的 num_kv_groups 个头，
# 每 group_size = num_heads // num_kv_groups 个 Query 头共享同一组 K/V，
# 从而大幅减少推理时 KV 缓存的显存占用。head_dim 可以独立于 d_in/num_heads 单独配置
# （Qwen3 各规格模型统一用 head_dim=128，与 emb_dim 解耦）。
# qk_norm=True 时会在 Q、K 上各加一个按 head_dim 归一化的 RMSNorm（即 QK-Norm），
# 用于稳定注意力的数值范围、缓解训练不稳定问题。
class GroupedQueryAttention(nn.Module):
    def __init__(
        self, d_in, num_heads, num_kv_groups, head_dim=None, qk_norm=False, dtype=None
    ):
        super().__init__()
        assert num_heads % num_kv_groups == 0, "num_heads must be divisible by num_kv_groups"

        self.num_heads = num_heads
        self.num_kv_groups = num_kv_groups
        self.group_size = num_heads // num_kv_groups

        # 中文注释：若未显式指定 head_dim，则退化为标准做法：head_dim = d_in // num_heads
        if head_dim is None:
            assert d_in % num_heads == 0, "`d_in` must be divisible by `num_heads` if `head_dim` is not set"
            head_dim = d_in // num_heads

        self.head_dim = head_dim
        self.d_out = num_heads * head_dim

        # 中文注释：Q 投影输出维度是 num_heads*head_dim；K、V 投影输出维度只有 num_kv_groups*head_dim，
        # 这正是 GQA 相比普通多头注意力节省参数量与显存的关键所在。
        self.W_query = nn.Linear(d_in, self.d_out, bias=False, dtype=dtype)
        self.W_key = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)
        self.W_value = nn.Linear(d_in, num_kv_groups * head_dim, bias=False, dtype=dtype)

        self.out_proj = nn.Linear(self.d_out, d_in, bias=False, dtype=dtype)

        # 中文注释：仅在 qk_norm=True 时才创建 QK-Norm 的两个 RMSNorm（作用维度是 head_dim）
        if qk_norm:
            self.q_norm = RMSNorm(head_dim, eps=1e-6)
            self.k_norm = RMSNorm(head_dim, eps=1e-6)
        else:
            self.q_norm = self.k_norm = None

    # 中文注释：forward 输入 x 形状 (b, num_tokens, d_in)，mask 为因果掩码，cos/sin 为 RoPE 表。
    def forward(self, x, mask, cos, sin):
        b, num_tokens, _ = x.shape

        # Apply projections
        queries = self.W_query(x)  # (b, num_tokens, num_heads * head_dim)
        keys = self.W_key(x)       # (b, num_tokens, num_kv_groups * head_dim)
        values = self.W_value(x)   # (b, num_tokens, num_kv_groups * head_dim)

        # Reshape
        # 中文注释：把最后一维拆成 (头数, head_dim)，并将头维度换到第 2 维，
        # 得到 queries: (b, num_heads, num_tokens, head_dim)；keys/values: (b, num_kv_groups, num_tokens, head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_kv_groups, self.head_dim).transpose(1, 2)

        # 中文注释：QK-Norm——对每个头单独做 RMSNorm（在 apply_rope 之前）
        # Optional normalization
        if self.q_norm:
            queries = self.q_norm(queries)
        if self.k_norm:
            keys = self.k_norm(keys)

        # 中文注释：对 Q、K 施加旋转位置编码（V 不需要）
        # Apply RoPE
        queries = apply_rope(queries, cos, sin)
        keys = apply_rope(keys, cos, sin)

        # 中文注释：用 repeat_interleave 把 K/V 的头数从 num_kv_groups 复制扩展到 num_heads，
        # 每组 K/V 被相邻的 group_size 个 Q 头重复使用，这就是 GQA“分组共享”的具体实现方式。
        # Expand K and V to match number of heads
        keys = keys.repeat_interleave(self.group_size, dim=1)
        values = values.repeat_interleave(self.group_size, dim=1)

        # 中文注释：标准缩放点积注意力：QK^T 后用因果掩码把“未来”位置填为 -inf，
        # 再除以 sqrt(head_dim) 做缩放并 softmax 得到注意力权重
        # Attention
        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores = attn_scores.masked_fill(mask, -torch.inf)
        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)

        # 中文注释：加权求和后把 (b, num_heads, num_tokens, head_dim) 转回 (b, num_tokens, d_out)，
        # 最后经过输出投影 out_proj 映射回 d_in 维度
        context = (attn_weights @ values).transpose(1, 2).reshape(b, num_tokens, self.d_out)
        return self.out_proj(context)
# 中文注释：Transformer 层，采用“前置归一化 + 残差连接”（Pre-Norm）结构：
# x -> norm1 -> 注意力 -> 加回残差 -> norm2 -> 前馈网络 -> 加回残差
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = GroupedQueryAttention(
            d_in=cfg["emb_dim"],
            num_heads=cfg["n_heads"],
            head_dim=cfg["head_dim"],
            num_kv_groups=cfg["n_kv_groups"],
            qk_norm=cfg["qk_norm"],
            dtype=cfg["dtype"]
        )
        self.ff = FeedForward(cfg)
        self.norm1 = RMSNorm(cfg["emb_dim"], eps=1e-6)
        self.norm2 = RMSNorm(cfg["emb_dim"], eps=1e-6)

    # 中文注释：输入/输出 x 形状均为 (batch_size, num_tokens, emb_dim)，两个子层各自独立残差相加
    def forward(self, x, mask, cos, sin):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x, mask, cos, sin)  # Shape [batch_size, num_tokens, emb_size]
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed-forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = x + shortcut  # Add the original input back

        return x
# 中文注释：Qwen3 整体模型：词嵌入 -> n_layers 层 TransformerBlock -> 最终 RMSNorm -> 输出线性头（词表 logits）。
# RoPE 的 cos/sin 表在初始化时按 context_length 预计算一次，所有层共享；
# 若 checkpoint 未提供独立的 lm_head 权重，输出头会与词嵌入权重绑定（tied embeddings，见加载权重部分）。
class Qwen3Model(nn.Module):
    def __init__(self, cfg):
        super().__init__()

        # Main model parameters
        # 中文注释：词嵌入表，形状 (vocab_size, emb_dim)
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"], dtype=cfg["dtype"])

        # 中文注释：用 ModuleList 存放各层
        self.trf_blocks = nn.ModuleList(  # ModuleList since Sequential can only accept one input, and we need `x, mask, cos, sin`
            [TransformerBlock(cfg) for _ in range(cfg["n_layers"])]
        )

        # 中文注释：最终输出前的归一化，以及把 emb_dim 映射到 vocab_size 的输出头（无偏置）
        self.final_norm = RMSNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False, dtype=cfg["dtype"])

        # 中文注释：提前算好 RoPE 的 cos/sin 表，注册为非持久 buffer
        # （persistent=False 表示不会被保存进 state_dict，每次构建模型都会用当前配置重新计算）
        # Reusable utilities
        if cfg["head_dim"] is None:
            head_dim = cfg["emb_dim"] // cfg["n_heads"]
        else:
            head_dim = cfg["head_dim"]
        cos, sin = compute_rope_params(
            head_dim=head_dim,
            theta_base=cfg["rope_base"],
            context_length=cfg["context_length"]
        )
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.cfg = cfg


    # 中文注释：forward 输入 in_idx 形状 (batch, seq_len) 的 token id；
    # 输出 logits 形状 (batch, seq_len, vocab_size)
    def forward(self, in_idx):
        # Forward pass
        tok_embeds = self.tok_emb(in_idx)
        x = tok_embeds

        num_tokens = x.shape[1]
        # 中文注释：构造下三角（含对角线可见）的因果注意力掩码：
        # torch.triu(..., diagonal=1) 得到严格上三角为 True，表示“未来位置”，会在注意力里被屏蔽
        mask = torch.triu(torch.ones(num_tokens, num_tokens, device=x.device, dtype=torch.bool), diagonal=1)

        for block in self.trf_blocks:
            x = block(x, mask, self.cos, self.sin)
        x = self.final_norm(x)
        logits = self.out_head(x.to(self.cfg["dtype"]))
        return logits

2. Initialize model

In [ ]:
# 中文注释：按不同参数规模（0.6B ~ 32B）定义 Qwen3 的超参数配置。
# 注意各规格都固定 head_dim=128、n_kv_groups=8，即随着 n_heads 增大，
# 每组 K/V 被更多 Query 头共享（group_size = n_heads / n_kv_groups 逐档变化）；
# qk_norm=True 表示都启用 QK-Norm；rope_base=1e6 配合 context_length=40960 支持长上下文。
CHOOSE_MODEL = "0.6B"

if CHOOSE_MODEL == "0.6B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,           # Vocabulary size
        "context_length": 40_960,        # Context length that was used to train the model
        "emb_dim": 1024,                 # Embedding dimension
        "n_heads": 16,                   # Number of attention heads
        "n_layers": 28,                  # Number of layers
        "hidden_dim": 3072,              # Size of the intermediate dimension in FeedForward
        "head_dim": 128,                 # Size of the heads in GQA
        "qk_norm": True,                 # Whether to normalize queries and keys in GQA
        "n_kv_groups": 8,                # Key-Value groups for grouped-query attention
        "rope_base": 1_000_000.0,        # The base in RoPE's "theta"
        "dtype": torch.bfloat16,         # Lower-precision dtype to reduce memory usage
    }

elif CHOOSE_MODEL == "1.7B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,
        "context_length": 40_960,
        "emb_dim": 2048,                 # 2x larger than above
        "n_heads": 16,
        "n_layers": 28,
        "hidden_dim": 6144,              # 2x larger than above
        "head_dim": 128,
        "qk_norm": True,
        "n_kv_groups": 8,
        "rope_base": 1_000_000.0,
        "dtype": torch.bfloat16,
    }

elif CHOOSE_MODEL == "4B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,
        "context_length": 40_960,
        "emb_dim": 2560,                 # 25% larger than above
        "n_heads": 32,                   # 2x larger than above
        "n_layers": 36,                  # 29% larger than above
        "hidden_dim": 9728,              # ~3x larger than above
        "head_dim": 128,
        "qk_norm": True,
        "n_kv_groups": 8,
        "rope_base": 1_000_000.0,
        "dtype": torch.bfloat16,
    }

elif CHOOSE_MODEL == "8B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,
        "context_length": 40_960,
        "emb_dim": 4096,                 # 60% larger than above
        "n_heads": 32,
        "n_layers": 36,                  # 26% larger than above
        "hidden_dim": 12288,
        "head_dim": 128,
        "qk_norm": True,
        "n_kv_groups": 8,
        "rope_base": 1_000_000.0,
        "dtype": torch.bfloat16,
    }

elif CHOOSE_MODEL == "14B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,
        "context_length": 40_960,
        "emb_dim": 5120,                 # 25% larger than above
        "n_heads": 40,                   # 25% larger than above
        "n_layers": 40,                  # 11% larger than above
        "hidden_dim": 17408,             # 42% larger than above
        "head_dim": 128,
        "qk_norm": True,
        "n_kv_groups": 8,
        "rope_base": 1_000_000.0,
        "dtype": torch.bfloat16,
    }

elif CHOOSE_MODEL == "32B":
    QWEN3_CONFIG = {
        "vocab_size": 151_936,
        "context_length": 40_960,
        "emb_dim": 5120,
        "n_heads": 64,                   # 60% larger than above
        "n_layers": 64,                  # 60% larger than above
        "hidden_dim": 25600,             # 47% larger than above
        "head_dim": 128,
        "qk_norm": True,
        "n_kv_groups": 8,
        "rope_base": 1_000_000.0,
        "dtype": torch.bfloat16,
    }

else:
    raise ValueError(f"{CHOOSE_MODEL} is not supported.")
# 中文注释：固定随机种子，使随机初始化的权重可复现（后面加载预训练权重后会被整体覆盖）
torch.manual_seed(123)
# 中文注释：用上面选定的配置构建一个随机初始化的 Qwen3Model 实例
model = Qwen3Model(QWEN3_CONFIG)
model

In [ ]:
# 中文注释：用一个简单的示例输入 (batch=1, seq_len=3) 做一次前向传播，快速验证模型结构能否正常跑通，
# 输出 logits 的形状应为 (1, 3, vocab_size)
model(torch.tensor([1, 2, 3]).unsqueeze(0))

In [ ]:
# 中文注释：统计模型中所有参数张量的元素总数（与 dtype 无关，只看元素个数）
total_params = sum(p.numel() for p in model.parameters())
print(f"Total number of parameters: {total_params:,}")

# 中文注释：0.6B/1.7B 等较小规格的 Qwen3 会采用“权重绑定”（tie embeddings），
# 即加载预训练权重后 out_head.weight 会直接复用 tok_emb.weight（同一份张量），
# 这会导致 sum(p.numel()) 把这份 embedding 参数重复计数一次，
# 这里减去一次 tok_emb 的元素数，得到“去重后的实际参数量”估计
# （此时权重尚未加载，这里是按最终会绑定的情况提前估算）
# Account for weight tying
total_params_normalized = total_params - model.tok_emb.weight.numel()
print(f"\nTotal number of unique parameters: {total_params_normalized:,}")

In [ ]:
# 中文注释：估算模型在给定 dtype 下的内存占用（参数 + 梯度 + 缓冲区），
# 用来判断某个精度（float32/bfloat16 等）下模型能否放进可用显存/内存
def calc_model_memory_size(model, input_dtype=torch.float32):
    total_params = 0
    total_grads = 0
    for param in model.parameters():
        # Calculate total number of elements per parameter
        param_size = param.numel()
        total_params += param_size
        # Check if gradients are stored for this parameter
        if param.requires_grad:
            total_grads += param_size

    # Calculate buffer size (non-parameters that require memory)
    # 中文注释：buffers 里包含了 RoPE 预计算出的 cos/sin 表（虽然 persistent=False，
    # 不会被存进 state_dict，但仍然占用运行时内存，因此这里一并统计）
    total_buffers = sum(buf.numel() for buf in model.buffers())

    # Size in bytes = (Number of elements) * (Size of each element in bytes)
    # We assume parameters and gradients are stored in the same type as input dtype
    element_size = torch.tensor(0, dtype=input_dtype).element_size()
    total_memory_bytes = (total_params + total_grads + total_buffers) * element_size

    # Convert bytes to gigabytes
    total_memory_gb = total_memory_bytes / (1024**3)

    return total_memory_gb

# 中文注释：对比全精度 float32 与低精度 bfloat16 下模型各自需要的内存大小
print(f"float32 (PyTorch default): {calc_model_memory_size(model, input_dtype=torch.float32):.2f} GB")
print(f"bfloat16: {calc_model_memory_size(model, input_dtype=torch.bfloat16):.2f} GB")

In [ ]:
# 中文注释：按优先级选择计算设备：CUDA GPU > Apple Silicon 的 MPS > CPU
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# 中文注释：把模型所有参数和缓冲区（包括 RoPE 的 cos/sin 表）转移到所选设备上；
# 末尾分号用于抑制该单元格的自动输出显示
model.to(device);

3. Load pretrained weights

In [ ]:
# 中文注释：把 Hugging Face 官方发布的 Qwen3 safetensors checkpoint 中按名称存放的参数，
# 逐一映射拷贝到我们自定义模型对应的张量里，并在拷贝前做形状校验。
def load_weights_into_qwen(model, param_config, params):
    # 中文注释：assign 是一个小工具函数：先检查左右两边形状是否一致，
    # 再用 copy_ 原地写入（不改变原 nn.Parameter 对象本身，只替换其中的数值）
    def assign(left, right, tensor_name="unknown"):
        if left.shape != right.shape:
            raise ValueError(f"Shape mismatch in tensor '{tensor_name}'. Left: {left.shape}, Right: {right.shape}")

        with torch.no_grad():
            if isinstance(right, torch.Tensor):
                left.copy_(right)
            else:
                left.copy_(torch.as_tensor(right, dtype=left.dtype, device=left.device))

        return left

    # 中文注释：加载词嵌入表，形状 (vocab_size, emb_dim)
    model.tok_emb.weight = assign(model.tok_emb.weight, params["model.embed_tokens.weight"], "model.embed_tokens.weight")

    # 中文注释：逐层遍历，按 HF 官方命名规则取出该层的 Q/K/V/O 投影、
    # 可选的 QK-Norm、两个 RMSNorm，以及 SwiGLU 的三个投影权重
    for l in range(param_config["n_layers"]):
        block = model.trf_blocks[l]
        att = block.att

        # 中文注释：q_proj 权重形状对应 (num_heads*head_dim, d_in)；
        # k_proj/v_proj 权重形状对应 (num_kv_groups*head_dim, d_in) —— 体现 GQA 中 K/V 头数少于 Q 头数
        # Q, K, V projections
        att.W_query.weight = assign(
            att.W_query.weight,
            params[f"model.layers.{l}.self_attn.q_proj.weight"],
            f"model.layers.{l}.self_attn.q_proj.weight"
        )
        att.W_key.weight = assign(
            att.W_key.weight,
            params[f"model.layers.{l}.self_attn.k_proj.weight"],
            f"model.layers.{l}.self_attn.k_proj.weight"
        )
        att.W_value.weight = assign(
            att.W_value.weight,
            params[f"model.layers.{l}.self_attn.v_proj.weight"],
            f"model.layers.{l}.self_attn.v_proj.weight"
        )

        # Output projection
        att.out_proj.weight = assign(
            att.out_proj.weight,
            params[f"model.layers.{l}.self_attn.o_proj.weight"],
            f"model.layers.{l}.self_attn.o_proj.weight"
        )

        # 中文注释：QK-Norm 权重（RMSNorm 的 scale，形状为 head_dim），
        # 只有当模型配置 qk_norm=True、且 checkpoint 中确实存在该权重时才会被赋值
        # QK norms
        if hasattr(att, "q_norm") and att.q_norm is not None:
            att.q_norm.scale = assign(
                att.q_norm.scale,
                params[f"model.layers.{l}.self_attn.q_norm.weight"],
                f"model.layers.{l}.self_attn.q_norm.weight"
            )
        if hasattr(att, "k_norm") and att.k_norm is not None:
            att.k_norm.scale = assign(
                att.k_norm.scale,
                params[f"model.layers.{l}.self_attn.k_norm.weight"],
                f"model.layers.{l}.self_attn.k_norm.weight"
            )

        # Attention layernorm
        block.norm1.scale = assign(
            block.norm1.scale,
            params[f"model.layers.{l}.input_layernorm.weight"],
            f"model.layers.{l}.input_layernorm.weight"
        )

        # 中文注释：HF 权重命名里的 gate_proj / up_proj / down_proj
        # 分别对应本实现 FeedForward 里的 fc1 / fc2 / fc3，
        # 组成 SwiGLU：down_proj( silu(gate_proj(x)) * up_proj(x) )
        # Feedforward weights
        block.ff.fc1.weight = assign(
            block.ff.fc1.weight,
            params[f"model.layers.{l}.mlp.gate_proj.weight"],
            f"model.layers.{l}.mlp.gate_proj.weight"
        )
        block.ff.fc2.weight = assign(
            block.ff.fc2.weight,
            params[f"model.layers.{l}.mlp.up_proj.weight"],
            f"model.layers.{l}.mlp.up_proj.weight"
        )
        block.ff.fc3.weight = assign(
            block.ff.fc3.weight,
            params[f"model.layers.{l}.mlp.down_proj.weight"],
            f"model.layers.{l}.mlp.down_proj.weight"
        )
        block.norm2.scale = assign(
            block.norm2.scale,
            params[f"model.layers.{l}.post_attention_layernorm.weight"],
            f"model.layers.{l}.post_attention_layernorm.weight"
        )

    # 中文注释：最终归一化层，以及输出头（词表 logits 投影）
    # Final normalization and output head
    model.final_norm.scale = assign(model.final_norm.scale, params["model.norm.weight"], "model.norm.weight")

    # 中文注释：判断 checkpoint 是否包含独立的 lm_head 权重；
    # 若不存在（常见于 0.6B/1.7B 等较小规格），说明模型采用“权重绑定”（tied embeddings），
    # 输出头直接复用词嵌入权重，而非独立训练的一份参数
    if "lm_head.weight" in params:
        model.out_head.weight = assign(model.out_head.weight, params["lm_head.weight"], "lm_head.weight")
    else:
        model.out_head.weight = model.tok_emb.weight
        print("Model uses weight tying.")
# 中文注释：以下代码从 Hugging Face Hub 下载对应规格 Qwen3 的预训练权重文件，
# 根据是单文件 safetensors 还是分片（配 index.json）选择不同的下载/加载方式，
# 最终调用上面的 load_weights_into_qwen 把权重写入模型
import json
import os
from pathlib import Path
from safetensors.torch import load_file
from huggingface_hub import hf_hub_download, snapshot_download


# 中文注释：reasoning / instruct 模型使用不带 “-Base” 后缀的仓库（对话/推理微调版本）；
# 否则使用带 “-Base” 后缀的预训练基座模型仓库
if USE_REASONING_MODEL or USE_INSTRUCT_MODEL:
    repo_id = f"Qwen/Qwen3-{CHOOSE_MODEL}"
else:
    repo_id = f"Qwen/Qwen3-{CHOOSE_MODEL}-Base"

# 中文注释：取仓库名最后一段作为本地缓存目录名，例如 "Qwen3-0.6B" 或 "Qwen3-0.6B-Base"
local_dir = Path(repo_id).parts[-1]

# 中文注释：0.6B 模型体积较小，官方以单个 model.safetensors 文件发布，可直接下载；
# 其余更大规格被切分成多个分片文件，需要先下载并解析索引文件才能知道分片列表
if CHOOSE_MODEL == "0.6B":
    weights_file = hf_hub_download(
        repo_id=repo_id,
        filename="model.safetensors",
        local_dir=local_dir,
    )
    weights_dict = load_file(weights_file)
else:
    # 中文注释：下载整份仓库快照（因为分片文件名不固定，无法像上面一样只下载单个文件）
    repo_dir = snapshot_download(repo_id=repo_id, local_dir=local_dir)
    index_path = os.path.join(repo_dir, "model.safetensors.index.json")
    with open(index_path, "r") as f:
        index = json.load(f)

    # 中文注释：按索引文件里列出的所有分片名，逐个加载并合并进统一的权重字典
    weights_dict = {}
    for filename in set(index["weight_map"].values()):
        shard_path = os.path.join(repo_dir, filename)
        shard = load_file(shard_path)
        weights_dict.update(shard)

# 中文注释：把下载并合并好的权重字典写入模型，再整体搬到目标设备，
# 最后删除 CPU 上的临时字典以释放内存
load_weights_into_qwen(model, QWEN3_CONFIG, weights_dict)
model.to(device)
del weights_dict

4. Load tokenizer

In [ ]:
# 中文注释：Qwen3Tokenizer 封装了 tokenizers 库的 BPE 分词器，
# 并额外处理 Qwen3 的一系列特殊 token、以及 chat 模板的拼装
import re
from tokenizers import Tokenizer

# 中文注释：Qwen3 专用分词器封装类
class Qwen3Tokenizer:
    # 中文注释：Qwen3 词表中的保留特殊 token：包括对话边界 <|im_start|>/<|im_end|>、
    # 多模态占位符，以及推理模型专用的 <think></think> 思考标签
    _SPECIALS = [
        "<|endoftext|>",
        "<|im_start|>", "<|im_end|>",
        "<|object_ref_start|>", "<|object_ref_end|>",
        "<|box_start|>", "<|box_end|>",
        "<|quad_start|>", "<|quad_end|>",
        "<|vision_start|>", "<|vision_end|>",
        "<|vision_pad|>", "<|image_pad|>", "<|video_pad|>",
        "<think>", "</think>"
    ]
    # 中文注释：用正则把文本中的特殊 token 与普通文本分离，
    # 普通文本片段再交给底层 BPE 做子词切分，特殊 token 整体映射为对应 id，避免被切碎
    _SPLIT_RE = re.compile(r"(<\|[^>]+?\|>|<think>|</think>)")

    # 中文注释：apply_chat_template 是否套用对话模板；add_generation_prompt 是否在末尾
    # 追加“assistant”生成提示；add_thinking 是否允许模型自己输出思考过程（仅 reasoning 模型用）
    def __init__(self, tokenizer_file_path="tokenizer.json", repo_id=None,
                 apply_chat_template=True, add_generation_prompt=False, add_thinking=False):

        self.apply_chat_template = apply_chat_template
        self.add_generation_prompt = add_generation_prompt
        self.add_thinking = add_thinking

        # 中文注释：这里的 Path 复用了前面加载预训练权重那个单元格中
        # `from pathlib import Path` 的导入（notebook 按执行顺序共享全局命名空间），
        # 本单元格未重复 import，属于顺序依赖，不是 bug，但要求必须先运行过前面的单元格
        tok_file = Path(tokenizer_file_path)
        self._tok = Tokenizer.from_file(str(tok_file))
        self._special_to_id = {}
        for t in self._SPECIALS:
            tid = self._tok.token_to_id(t)
            if tid is not None:
                self._special_to_id[t] = tid

        # 中文注释：默认用 <|endoftext|> 作为 pad token，也先临时作为 eos token
        self.pad_token_id = self._special_to_id["<|endoftext|>"]
        self.eos_token_id = self.pad_token_id

        # 中文注释：对话/推理模型的真正结束符是 <|im_end|>；
        # base 模型没有 chat 模板，结束符退回 <|endoftext|>
        if repo_id and "Base" not in repo_id:
            eos_token = "<|im_end|>"
        else:
            eos_token = "<|endoftext|>"
        if eos_token in self._special_to_id:
            self.eos_token_id = self._special_to_id[eos_token]

    # 中文注释：把文本编码为 token id 列表（List[int]，不是张量）
    def encode(self, text, chat_wrapped=None):
        if chat_wrapped is None:
            chat_wrapped = self.apply_chat_template

        # 中文注释：若输入内容本身恰好就是某个特殊 token 字符串，直接原样映射为对应 id，
        # 不再做 chat 模板包装或子词切分
        stripped = text.strip()
        if stripped in self._special_to_id and "\n" not in stripped:
            return [self._special_to_id[stripped]]

        # 中文注释：按需套用 chat 模板（例如加上 <|im_start|>user ... <|im_end|>）
        if chat_wrapped:
            text = self._wrap_chat(text)

        ids = []
        for part in filter(None, self._SPLIT_RE.split(text)):
            if part in self._special_to_id:
                ids.append(self._special_to_id[part])
            else:
                ids.extend(self._tok.encode(part).ids)
        return ids

    # 中文注释：skip_special_tokens=False 会保留特殊 token 对应的文本，
    # 便于观察模型是否正确生成了 <think>、<|im_end|> 等标记
    def decode(self, ids):
        return self._tok.decode(ids, skip_special_tokens=False)

    # 中文注释：拼装单轮对话的 chat 模板文本：
    # <|im_start|>user\n{内容}<|im_end|>\n，若需要生成提示则再追加 <|im_start|>assistant；
    # 当 add_thinking=False 时，额外插入一段空的 <think>\n\n</think> 强制关闭思考模式
    def _wrap_chat(self, user_msg):
        s = f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        if self.add_generation_prompt:
            s += "<|im_start|>assistant"
            if self.add_thinking:
                s += "\n"
            else:
                s += "\n<think>\n\n</think>\n\n"
        return s
# 中文注释【bug 修复】：原代码此处只判断了 USE_REASONING_MODEL，
# 与前面下载权重那个单元格中 repo_id/local_dir 的判断逻辑，以及本单元格下方第 83 行
# `USE_REASONING_MODEL or USE_INSTRUCT_MODEL` 的判断不一致。
# 若 USE_INSTRUCT_MODEL=True 且 USE_REASONING_MODEL=False，会拼出带 "-Base" 后缀的
# tokenizer 路径，但实际下载目录 local_dir 并没有 "-Base" 后缀，导致 FileNotFoundError。
# 已按相同逻辑修正为 `USE_REASONING_MODEL or USE_INSTRUCT_MODEL`。
if USE_REASONING_MODEL or USE_INSTRUCT_MODEL:
    tokenizer_file_path = f"Qwen3-{CHOOSE_MODEL}/tokenizer.json"
else:
    tokenizer_file_path = f"Qwen3-{CHOOSE_MODEL}-Base/tokenizer.json"

# 中文注释：单独下载 tokenizer.json 文件（权重可能在上一单元格中已按需下载过）
hf_hub_download(
    repo_id=repo_id,
    filename="tokenizer.json",
    local_dir=local_dir,
)

# 中文注释：reasoning/instruct 模型需要套用 chat 模板并追加生成提示；
# reasoning 模型允许输出 <think>...</think>（add_thinking=True），
# instruct 模型则默认插入空的 think 块以直接关闭思考模式
if USE_REASONING_MODEL or USE_INSTRUCT_MODEL:
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_file_path,
        repo_id=repo_id,
        apply_chat_template=True,
        add_generation_prompt=True,
        add_thinking=USE_REASONING_MODEL
    )

# 中文注释：base 模型没有对话身份区分，不使用 chat 模板，直接编码原始文本
else:
    tokenizer = Qwen3Tokenizer(
        tokenizer_file_path=tokenizer_file_path,
        repo_id=repo_id,
        apply_chat_template=False,
        add_generation_prompt=False,
        add_thinking=False
    )

In [ ]:
# 中文注释：用一句英文 prompt 测试 tokenizer 的编码/解码是否互逆；
# 注意在 reasoning/instruct 模式下，decode 结果会包含 chat 模板拼接进去的特殊 token 文本
prompt = "Give me a short introduction to large language models."

input_token_ids = tokenizer.encode(prompt)
text = tokenizer.decode(input_token_ids)
text

4. Generate text

In [ ]:
# 中文注释：贪婪解码（greedy decoding）版本的流式文本生成器。
# 注意：这里没有使用 KV 缓存，每一步都会把迄今为止的完整序列重新喂给模型做前向传播，
# 只是用于演示生成流程，并非推理效率最优的实现
def generate_text_basic_stream(model, token_ids, max_new_tokens, eos_token_id=None):

    # 中文注释：切换到 eval 模式，关闭训练专用行为（本模型定义未显式使用 dropout，仍是标准写法）
    model.eval()
    with torch.no_grad():
        for _ in range(max_new_tokens):
            # 中文注释：前向传播得到 (batch, seq_len, vocab_size) 的 logits，
            # 只取最后一个位置（下一个待预测 token）的 logits，得到 (batch, vocab_size)
            out = model(token_ids)[:, -1]
            # 中文注释：贪婪采样——直接取概率最大（argmax）的 token id，形状 (batch, 1)
            next_token = torch.argmax(out, dim=-1, keepdim=True)

            # 中文注释：若生成了结束符则提前停止（batch 内所有样本都命中 eos 才停止）
            if (eos_token_id is not None
                   and torch.all(next_token == eos_token_id)):
               break

            # 中文注释：以生成器（generator）形式逐个 token yield 出去，便于外部流式打印
            yield next_token

            # 中文注释：把新生成的 token 拼接回输入序列，作为下一步的上下文
            # （因为没有 KV 缓存，下一步会把整段更长的序列重新完整送入模型）
            token_ids = torch.cat([token_ids, next_token], dim=1)
# 中文注释：实际运行一次文本生成，并统计生成速度（tokens/sec）与 GPU 显存峰值占用
import time

# 中文注释：把 encode 得到的 id 列表转成形状 (1, prompt_len) 的张量（batch_size=1），
# 并放到与模型相同的 device 上
input_token_ids_tensor = torch.tensor(input_token_ids, device=device).unsqueeze(0)

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()

start_time = time.perf_counter()
generated_tokens = 0

# 中文注释：逐 token 流式生成并立即解码打印，直到达到 max_new_tokens 或遇到 eos
for token in generate_text_basic_stream(
    model=model,
    token_ids=input_token_ids_tensor,
    max_new_tokens=500,
    eos_token_id=tokenizer.eos_token_id
):
    generated_tokens += 1
    token_id = token.squeeze(0).tolist()
    print(
        tokenizer.decode(token_id),
        end="",
        flush=True
    )

# 中文注释：统计整体耗时与生成速度
elapsed = time.perf_counter() - start_time
tokens_per_sec = generated_tokens / elapsed if elapsed > 0 else 0.0
print(f"\n\nGeneration speed: {tokens_per_sec:.2f} tokens/sec")

# 中文注释：仅在 CUDA 设备上统计峰值显存占用（MPS/CPU 没有对应的统计 API）
if torch.cuda.is_available():
    def calc_gpu_gb(x):
        return f"{x / 1024 / 1024 / 1024:.2f} GB"

    print(f"GPU memory used: {calc_gpu_gb(torch.cuda.max_memory_allocated())}")